# Decision Support System (DSS)
## Context-Aware Music Recommendation & Health-Routine Optimization

**Studente:** Stefano  
**Corso:** Decision Support Systems / Sistemi Basati su Conoscenza  

---

### Abstract del Progetto
I sistemi convenzionali di raccomandazione musicale (come Spotify, Apple Music o YouTube) soffrono del problema della **"raccomandazione cieca al contesto"**: propongono canzoni basandosi sullo storico di ascolto o sulla popolarità globale, ignorando completamente **come ci sentiamo** in un dato istante.

Questo **Decision Support System (DSS)** colma questa lacuna integrando:
1. **Caratteristiche acustico-matematiche di Spotify** (BPM, valenza emotiva, livello energetico, strumentalità).
2. **Telemetria biometrica real-time da Smartwatch** (frequenza cardiaca, livello di stress su scala 0-100, attività in corso).

Attraverso la **programmazione logica (Prolog)** e regole di matching fisiologico, il DSS seleziona la traccia ottimale o genera una sessione guidata di rilassamento (*Iso-Principle*) spiegando sempre il razionale clinico della scelta.

## 1. Caricamento ed Esplorazione dei Dataset
Il sistema legge i file sorgente in `data/` contenenti il catalogo musicale e i profili biometrici.

In [1]:
import json
from pathlib import Path

with open('data/spotify_tracks.json', 'r', encoding='utf-8') as f:
    tracks = json.load(f)
with open('data/smartwatch_biometrics.json', 'r', encoding='utf-8') as f:
    users = json.load(f)

print(f"[+] Caricati con successo {len(tracks)} brani Spotify e {len(users)} profili biometrici smartwatch.")

print("\n--- Esempio Utenti Smartwatch ---")
for u in users[:3]:
    print(f" * {u['name']} ({u['user_id']}) -> Attività: {u['current_activity']} | Battito: {u['heart_rate_bpm']} BPM | Stress: {u['stress_level']}/100")


[+] Caricati con successo 30 brani Spotify e 6 profili biometrici smartwatch.

--- Esempio Utenti Smartwatch ---
 * Sarah Miller (user_sarah) -> Attività: high_stress_work | Battito: 114.0 BPM | Stress: 84.0/100
 * Alex Chen (user_alex) -> Attività: cardio_running | Battito: 146.0 BPM | Stress: 45.0/100
 * Marco Rossi (user_marco) -> Attività: deep_focus_study | Battito: 74.0 BPM | Stress: 32.0/100


## 2. Generazione Automatica della Knowledge Base Prolog
Lo script `generate_kb.py` traduce i file JSON in una Knowledge Base relazionale normalizzata (`knowledge_base.pl`) contenente i predicati `brano/8` e `utente/7`.

In [2]:
# Esecuzione dell'ETL di compilazione della Knowledge Base
import generate_kb
generate_kb.main()

# Ispezione delle prime 16 righe di fatti Prolog generati
with open('knowledge_base.pl', 'r', encoding='utf-8') as f:
    righe = [next(f) for _ in range(16)]
print(''.join(righe))


[*] Caricamento dataset JSON...
[*] Generazione fatti Prolog in knowledge_base.pl...
[+] Generazione completata con successo: 30 brani e 6 profili utente scritti in knowledge_base.pl.

% =========================================================================
% KNOWLEDGE BASE PROLOG: Decision Support System (DSS)
% Generato automaticamente da generate_kb.py a partire dai dati Spotify e Smartwatch
% =========================================================================

% --- Fatti: brano(Id, Titolo, Artista, Genere, BPM, Valenza, Energia, Strumentale) ---
:- dynamic brano/8.

brano(weightless_ambient, 'Weightless Ambient Drift', 'NeuroHarmonics', ambient, 60.0, 0.35, 0.12, 0.96).
brano(clair_de_lune, 'Clair de Lune (Debussy)', 'Sinfonia Chambers', classical, 64.0, 0.42, 0.18, 0.94).
brano(delta_sleep_stream, 'Delta Wave Binaural Serenade', 'SomnoLab', ambient, 55.0, 0.25, 0.08, 0.99).
brano(gymnopedie_no1, 'Gymnopédie No. 1 (Satie)', 'Aura Piano Quartet', classical, 66.0, 0.38, 0.1

## 3. Le Regole del Decision Support System (`dss_rules.pl`)
Il motore di inferenza applica tre vincoli principali per filtrare i brani:
1. **Vincolo Cognitivo (Studio):** Nessun brano con parole cantate (`instrumentalness >= 0.70`), azzerando le interferenze con la memoria di lavoro verbale.
2. **Vincolo Motorio (Corsa):** Brani con `BPM >= 120` e `Energia >= 0.75` sincronizzati col passo di corsa per ridurre la fatica percepita (*Rate of Perceived Exertion*).
3. **Vincolo di Rilassamento (Stress):** Brani a basso tempo (`BPM <= 75`) e bassa energia per stimolare il tono vagale e down-regolare il sistema nervoso simpatico.

In [3]:
from test_suite import load_data, raccomanda

tracks_map, users_map = load_data()

scenari = [
    ('user_sarah', 'CASO 1: Stress Lavorativo Acuto'),
    ('user_alex',  'CASO 2: Allenamento Corsa Cardio'),
    ('user_marco', 'CASO 3: Sessione Studio / Deep Focus')
]

for uid, titolo in scenari:
    u = users_map[uid]
    best_track, punteggio = raccomanda(uid, tracks_map, users_map)
    print('=' * 72)
    print(f' {titolo}')
    print('=' * 72)
    print(f' Utente: {u["name"]} | Attività: {u["current_activity"]}')
    print(f' Telemetria: Battito: {u["heart_rate_bpm"]} BPM | Stress: {u["stress_level"]}/100')
    print(f'\n -> BRANO RACCOMANDATO: \'{best_track["title"]}\' di {best_track["artist"]}')
    print(f' -> Punteggio Utilità: {punteggio:.1f}/100 | Genere: {best_track["genre"]}')
    print(f' -> Parametri: {best_track["bpm"]} BPM, Energia: {best_track["energy"]}, Strumentale: {best_track["instrumentalness"]}')
    print()


 CASO 1: Stress Lavorativo Acuto
 Utente: Sarah Miller | Attività: high_stress_work
 Telemetria: Battito: 114.0 BPM | Stress: 84.0/100

 -> BRANO RACCOMANDATO: 'Peaceful Mind Breeze' di Zenith Soundscapes
 -> Punteggio Utilità: 90.1/100 | Genere: ambient
 -> Parametri: 68.0 BPM, Energia: 0.22, Strumentale: 0.91

 CASO 2: Allenamento Corsa Cardio
 Utente: Alex Chen | Attività: cardio_running
 Telemetria: Battito: 146.0 BPM | Stress: 45.0/100

 -> BRANO RACCOMANDATO: 'Subterranean Stamina Loop' di Klangwerk
 -> Punteggio Utilità: 88.7/100 | Genere: techno
 -> Parametri: 140.0 BPM, Energia: 0.92, Strumentale: 0.9

 CASO 3: Sessione Studio / Deep Focus
 Utente: Marco Rossi | Attività: deep_focus_study
 Telemetria: Battito: 74.0 BPM | Stress: 32.0/100

 -> BRANO RACCOMANDATO: 'Raindrops on University Library' di Coffeehouse Chill
 -> Punteggio Utilità: 90.8/100 | Genere: lofi
 -> Parametri: 76.0 BPM, Energia: 0.34, Strumentale: 0.89



## 4. Altshuler's Iso-Principle (Musicoterapia Applicata)
Quando una persona ha un battito cardiaco elevato per agitazione (es. 114 BPM), passare bruscamente a un brano lento genera dissonanza affettiva e rigetto.

L'**Iso-Principle** adotta una traiettoria controllata a 3 scalini:
- **Stadio 1 (Entry / Aggancio):** Incontra l'utente sul suo tempo attuale (105 BPM).
- **Stadio 2 (Bridge / Ponte):** Inizia la decelerazione controllata (85 BPM).
- **Stadio 3 (Target / Stabilizzazione):** Guida verso la frequenza a riposo (60 BPM).

In [4]:
from test_suite import test_iso_principle

t1, t2, t3 = test_iso_principle(tracks_map, users_map)

print('+======================================================================+')
print('| SEQUENZA MULTI-BRANO AD ENTRAINMENT (ISO-PRINCIPLE)                  |')
print('+======================================================================+')
print(f'| 1. [AGGANCIO]  {t1["title"]:28} ({t1["bpm"]:.0f} BPM, E:{t1["energy"]:.2f})             |')
print(f'| 2. [PONTE]     {t2["title"]:28} ({t2["bpm"]:.0f} BPM, E:{t2["energy"]:.2f})             |')
print(f'| 3. [OBIETTIVO] {t3["title"]:28} ({t3["bpm"]:.0f} BPM, E:{t3["energy"]:.2f})             |')
print('+----------------------------------------------------------------------+')
print(f'| Traiettoria: {t1["bpm"]:.0f} BPM -> {t2["bpm"]:.0f} BPM -> {t3["bpm"]:.0f} BPM (Rallentamento controllato)|')
print('+======================================================================+')


+======================================================================+
| SEQUENZA MULTI-BRANO AD ENTRAINMENT (ISO-PRINCIPLE)                  |
+======================================================================+
| 1. [AGGANCIO]  Iso Bridge Dynamic Transition (105 BPM, E:0.55)             |
| 2. [PONTE]     Tempo Taper Andante          (85 BPM, E:0.38)             |
| 3. [OBIETTIVO] Weightless Ambient Drift     (60 BPM, E:0.12)             |
+----------------------------------------------------------------------+
| Traiettoria: 105 BPM -> 85 BPM -> 60 BPM (Rallentamento controllato)|
+======================================================================+


## 5. Esecuzione Dedicata dello Script Esterno `test_suite.py`
In questa sezione conclusiva possiamo lanciare direttamente il file `.py` di test esterno ([`test_suite.py`](test_suite.py)).

In un ambiente Jupyter puoi eseguirlo direttamente con il comando magico `%run test_suite.py` (oppure con `import test_suite; test_suite.main()`).

La cella sottostante è **completamente pre-eseguita** e mostra l'intero output atteso con tutti i 10 test di correttezza logica superati:

In [5]:
# Esecuzione dello script Python esterno 'test_suite.py'
# (In Jupyter puoi usare anche il comando magico: %run test_suite.py)
import test_suite
test_suite.main()



 VERIFICA DI CORRETTEZZA LOGICA DEL SISTEMA (TEST SUITE)
 [+] 1. Riduzione stress (Sarah Miller) -> Brano lento e rilassante: PASSATO
 [+] 2. Cardio Workout (Alex Chen) -> Cadenza veloce ed alta energia: PASSATO
 [+] 3. Deep Focus (Marco Rossi) -> Zero parole cantate (strumentale): PASSATO
 [+] 4. Sonno (David Becker) -> Tempo lentissimo e minima energia: PASSATO
 [+] 5. Recupero post workout (Elena Rostova) -> Tempo controllato: PASSATO
 [+] 6. Commute attivo (Clara Dupont) -> Tono allegro (valenza positiva): PASSATO
 [+] 7. Entrainment Iso-Principle -> Traiettoria monotonica decrescente: PASSATO
 [+] 8. Test Negativo -> Brano con parole escluso in fase di studio: PASSATO
 [+] 9. Test Negativo -> Brano lento escluso durante la corsa cardio: PASSATO
 [+] 10. Scoring -> Bonus per genere musicale preferito dall'utente: PASSATO
----------------------------------------------------------------------
 Esito Test: 10/10 Superati con successo.

